# 同行5分钟上下文补齐
只延展现有QQQ/SMH/NVDA/AMD上下文至核心共同日期，NONE/RTH，原前缀不改。完整日历审计、原生Qlib读回后发布NAS，0模型拟合、0回测。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/semis_context_refresh_20261014'
qlib.init(provider_uri=str(root/'data/qlib/soxl_intraday_1min'),region='us',kernels=1)
from src.workflow.semis_context_refresh import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'semis_context_refresh_20261014',
 'plan_sha256': 'b4900a3902c76a4e344a15c6ba9e264088cb0362f102dfc6424c774d8de32024',
 'decision': 'core_and_context_source_dates_aligned_feature_pipeline_not_refit',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'before_source_id': '149ef504c382386397944b1e7af203b3eace9d2c9e43e09d9164c9a3691b302c',
 'after_source_id': '2c70bd56ffc15ad4ba4eb243e0ca0b9409cd9d39f369c0d95ac02829de1fcf58',
 'core_as_of': '2026-10-13',
 'context_as_of': '2026-10-13',
 'new_context_bars': 624,
 'context_rows': 624672,
 'source_counts': [{'symbol': 'QQQ',
   'old_rows': 156012,
   'new_rows': 156,
   'merged_rows': 156168},
  {'symbol': 'SMH',
   'old_rows': 156012,
   'new_rows': 156,
   'merged_rows': 156168},
  {'symbol': 'NVDA',
   'old_rows': 156012,
   'new_rows': 156,
   'merged_rows': 156168},
  {'symbol': 'AMD',
   'old_rows': 156012,
   'new_rows': 156,
   'merged_rows': 156168}],
 'missing_context_bars': 0,
 'context_invalid_rows'

# 同行上下文补齐至2026-10-13

复用已有四个SOXL研究上下文标的，获取QQQ/SMH/NVDA/AMD在10月12–13日的K_5M、NONE、RTH原始数据。每标的156条，合计新增624条，未增加交易品种或修改策略。0拟合、0回测。

|标的|旧行数|新增|新行数|
|---|---:|---:|---:|
|QQQ|156012|156|156168|
|SMH|156012|156|156168|
|NVDA|156012|156|156168|
|AMD|156012|156|156168|

四标的共624672条5分钟原始记录。完整原生交易日历核对缺失0、意外分钟0、无效OHLC0；新两日每标的各78条。Qlib原生读回新增624条价格及有效标志通过。所有旧同行原始行逐值不变，原始与新尾部获取时间分别保留，没有用新时间覆盖旧来源记录。

核心1分钟原始缓存和核心复权事件未改。新共同来源仍保留canonical核心格式，在新清单位置保存其所需的核心事件文件；实际核心加载器已验证可读取。同行有单独的规范化NONE缓存、事件路径和5分钟provider，通过NAS binding的`context_*`元数据定位，而非修改旧冻结路径。

NAS旧版本`149ef504c382386397944b1e7af203b3eace9d2c9e43e09d9164c9a3691b302c`，新版本`2c70bd56ffc15ad4ba4eb243e0ca0b9409cd9d39f369c0d95ac02829de1fcf58`。新快照远端哈希与current登记回读通过，旧快照保留。核心与同行的来源日期均为10月13日；核心2020年的1条历史OHLC异常仍保留，整个来源不冒充完全无异常。

为防止后续只刷新核心时丢失同行来源，核心刷新器保留`context_*`元数据，并把同行无效行计入总体错误状态。既有模拟发布测试覆盖来源指针保留和同行错误不被核心成功掩盖。新增5分钟日历测试覆盖半日市、缺bar与异常日期。最终4项相关检查通过；新增的日期防护在当前完整数据上再次只读验证，不重复采集或发布。

决定：**core_and_context_source_dates_aligned_feature_pipeline_not_refit**。这是数据源对齐，不是所有日频/PIT/模型特征矩阵都已重建；`all_model_inputs_rebuilt=false`。历史行情补采、复权与原始实时首收/真实分红/券商成交仍有不同资格，不能把补采当作独立前向证据。

下一步应使特征构建明确读取这份共同NAS binding、验证所有输入可用时点与缺失掩码，再启动新的冻结研究或前向观测；不自动重启已失败的参数路线。稳定净SOXX超额目标未完成，生产release SHA及模型参数未改。

[来源行数](source_counts.csv)、[新增会话覆盖](latest_coverage.csv)、[Recorder清单](verification_manifest.json)。原始尾部、完整上下文缓存、复权事件与原生provider均由NAS版本持久保存；本地是缓存，Git只交付代码与瘦卡。
